# 4. Matrices de transición — cohorte unificada

A partir de **dataset/epocas_unificado.csv**, calcula matrices de transición 5×5 (Vigilia, S1, S2, SWS, REM) en dos normalizaciones complementarias, base del análisis de Markov:

- **Globales** — cada celda sobre el total de transiciones (toda la matriz suma 1). Una por paciente, una por dataset y una unificada (**todos**), en **imagenes/matrices/globales/**.
- **Por renglón** — probabilidad condicional **P(destino | origen)**, cada renglón suma 1. Misma cobertura, en **imagenes/matrices/renglon/**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path("..").resolve()
RUTA_CSV = RAIZ / "dataset/epocas_unificado.csv"
OUT_GLOBAL  = RAIZ / "imagenes/matrices/globales"
OUT_RENGLON = RAIZ / "imagenes/matrices/renglon"
for sub in ("por_paciente", "agrupadas"):
    (OUT_GLOBAL/sub).mkdir(parents=True, exist_ok=True)
    (OUT_RENGLON/sub).mkdir(parents=True, exist_ok=True)

FASES = ["Vigilia", "S1", "S2", "SWS", "REM"]
N = len(FASES)

df = pd.read_csv(RUTA_CSV)
df = df[df['fase_num'].between(0, 4)].copy()  # excluye Sin_clasificar
print("Pacientes:", df['paciente'].nunique(), "| Épocas:", len(df))

Pacientes: 127 | Épocas: 110950


In [2]:
def calcular_matriz_conteos(epocas_por_paciente):
    """Suma conteos de transición sobre uno o varios pacientes."""
    M = np.zeros((N, N), dtype=float)
    for _, g in epocas_por_paciente:
        seq = g.sort_values('epoca')['fase_num'].to_numpy()
        for a, b in zip(seq[:-1], seq[1:]):
            M[int(a), int(b)] += 1
    return M

def graficar_matriz(M, titulo, out_path, modo):
    """modo='global': normaliza por suma total. modo='renglon': por filas."""
    if modo == 'global':
        s = M.sum()
        P = M / s if s > 0 else M
        suma_real = P.sum()
    else:
        row_sum = M.sum(axis=1, keepdims=True)
        with np.errstate(invalid='ignore', divide='ignore'):
            P = np.where(row_sum > 0, M / row_sum, 0.0)
        suma_real = None

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(P, cmap='Blues')
    for i in range(N):
        for j in range(N):
            v = P[i, j]
            txt = '0' if v == 0 else (f'<0.001' if v < 0.001 else f'{v:.3f}')
            ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                    color='white' if v > P.max()*0.6 else 'black')
    ax.set_xticks(range(N)); ax.set_xticklabels(FASES)
    ax.set_yticks(range(N)); ax.set_yticklabels(FASES)
    ax.set_xlabel('Fase destino'); ax.set_ylabel('Fase origen')
    sub = f"\nSuma real = {suma_real:.6f}" if suma_real is not None else ""
    ax.set_title(titulo + sub)
    fig.colorbar(im, ax=ax, shrink=0.85)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

In [3]:
# --- Matrices por paciente ---
for paciente, g in df.groupby('paciente'):
    M = calcular_matriz_conteos([(paciente, g)])
    graficar_matriz(M, f"Matriz global - {paciente}", OUT_GLOBAL  / "por_paciente" / f"global_{paciente}.png", 'global')
    graficar_matriz(M, f"Matriz por renglón - {paciente}", OUT_RENGLON / "por_paciente" / f"renglon_{paciente}.png", 'renglon')

# --- Matrices por dataset ---
for ds, gds in df.groupby('dataset'):
    M = calcular_matriz_conteos(gds.groupby('paciente'))
    graficar_matriz(M, f"Matriz global - dataset {ds}", OUT_GLOBAL  / "agrupadas" / f"global_dataset_{ds}.png", 'global')
    graficar_matriz(M, f"Matriz por renglón - dataset {ds}", OUT_RENGLON / "agrupadas" / f"renglon_dataset_{ds}.png", 'renglon')

# --- Matriz unificada (todos los pacientes) ---
M_todos = calcular_matriz_conteos(df.groupby('paciente'))
graficar_matriz(M_todos, "Matriz global - todos los pacientes", OUT_GLOBAL  / "agrupadas" / "global_todos.png", 'global')
graficar_matriz(M_todos, "Matriz por renglón - todos los pacientes", OUT_RENGLON / "agrupadas" / "renglon_todos.png", 'renglon')

print("Globales en:", OUT_GLOBAL,  "|", len(list(OUT_GLOBAL.glob('*.png'))), "PNG")
print("Renglón  en:", OUT_RENGLON, "|", len(list(OUT_RENGLON.glob('*.png'))), "PNG")

Globales en: imagenes/matrices/globales | 0 PNG
Renglón  en: imagenes/matrices/renglon | 0 PNG


## Matriz global sin diagonal (todos los pacientes)

Las auto-transiciones (la diagonal) dominan los conteos y opacan la estructura de cambios entre fases. Esta versión pone la diagonal a cero y **renormaliza sobre el total de transiciones entre fases distintas**, para resaltar la dinámica de cambios reales. Se guarda en **imagenes/matrices/globales/agrupadas/**.

In [ ]:
# --- Matriz global SIN diagonal (todos los pacientes) ---
# Reutiliza M_todos (conteos de la matriz unificada): pone la diagonal a 0 y
# normaliza por la suma total de transiciones entre fases distintas.
M_sin_diag = M_todos.copy()
np.fill_diagonal(M_sin_diag, 0.0)

graficar_matriz(
    M_sin_diag,
    "Matriz global sin diagonal - todos los pacientes",
    OUT_GLOBAL / "agrupadas" / "global_sin_diagonal_todos.png",
    'global',
)
print("Guardada:", OUT_GLOBAL / "agrupadas" / "global_sin_diagonal_todos.png")

In [ ]:
# --- Matriz por renglón SIN diagonal (todos los pacientes) ---
# Pone la diagonal a 0 y normaliza por filas: dado que se abandona la fase de
# origen, ¿hacia qué fase se transita? (cada renglón suma 1).
graficar_matriz(
    M_sin_diag,
    "Matriz por renglón sin diagonal - todos los pacientes",
    OUT_RENGLON / "agrupadas" / "renglon_sin_diagonal_todos.png",
    'renglon',
)
print("Guardada:", OUT_RENGLON / "agrupadas" / "renglon_sin_diagonal_todos.png")